# TEST

## Multilingual-e5-large

In [1]:
import json, re
import chromadb
from langchain_openai.embeddings import OpenAIEmbeddings
from langchain_chroma import Chroma
from uuid import uuid4
import markdown2
from transformers import AutoTokenizer
import os
from concurrent.futures import ThreadPoolExecutor
from tqdm.autonotebook import tqdm

In [2]:
client = chromadb.HttpClient(
    host="3.35.104.197",
    port=10090
    )

In [3]:
client.list_collections()

['additional_docs',
 'multilingual-e5-large',
 'test',
 'HyQE_1',
 'HyQE',
 'hyqe',
 'data',
 'bge-m3',
 'benchmark_data']

In [25]:
embedding_function = OpenAIEmbeddings(
    base_url = "https://76mqtyy2wie64y-10050.proxy.runpod.net/v1",
    model = "multilingual-e5-large",
    api_key="team3",
    tiktoken_enabled=False,
    embedding_ctx_length = 502
)

chroma = Chroma(
                collection_name="multilingual", 
                embedding_function=embedding_function, 
                client=client
            )

In [ ]:
def insert_db(folder_path, chroma, tokenizer_model,  max_tokens=None, batch_size = 16):
    """
    ChromaDB 서버에 문서를 넣는 함수

    :param folderpath: 문서들이 있는 가장 상위 폴더 경로
    :param chroma: ChromaDB 클라이언트 객체
    :param tokenizer_model: 토크나이저 모델이름
    :param max_tokens:  None - 전체저장
                        max_tokens=512 - 512토큰단위 청킹 적용
    """
    texts=[]
    metadatas=[]
    tokenizer = AutoTokenizer.from_pretrained(tokenizer_model)
    for root, _, files in os.walk(folder_path):
        for filename in files:
            if filename.endswith((".txt", ".md", ".json")):
                file_path = os.path.join(root, filename)
    
                with open(file_path, "r", encoding="utf-8") as file:
                    content = file.read().strip()
    
                    if filename.endswith(".md"):
                        content = markdown2.markdown(content)
                    
                    elif filename.endswith(".json"):
                        try:
                            json_data = json.loads(content)
                            
                            if isinstance(json_data,list):
                                for item in  json_data:
                                    filename = item.get("title", "")
                                    content = item.get("content", "")
                            else:
                                
                                continue
                        except json.JSONDecodeError:
                            
                            continue

                    texts.append(content)
                    metadatas.append({
                        "source": filename,
                        "doc_id": str(uuid4())
                    })

    num_batches = len(texts) // batch_size +1 if len(texts) % batch_size else len(texts) // batch_size
    for i in tqdm(range(num_batches)):
        batch_text, batch_metadata = texts[i*batch_size:(i+1)*batch_size], metadatas[i*batch_size:(i+1)*batch_size]           
        chroma.add_texts(texts=batch_text, metadatas=batch_metadata)

In [7]:
insert_db(
    folder_path = r"C:\Users\sang\Desktop\Final_Project\DB",
    chroma = chroma,
    tokenizer_model= "multilingual-e5-large",
    max_tokens = 502
)

In [10]:
with open(r"C:\Users\sang\Desktop\Final_Project\benchmark_data_1000_new.json", "r", encoding="utf-8" )as f:
    benchmark_data = json.load(f)

In [ ]:
def hitratio_calculator(chroma, dataset, query_col, context_col, k_values):
    """
    ChromaDB 서버에서 검색 후 Hit Ratio를 계산하는 함수 (배치 처리 제거)

    :param chroma: ChromaDB 클라이언트 객체
    :param dataset: 데이터셋 (list of dicts)
    :param context_col: 정답이 들어있는 컬럼명 (예: "context")
    :param query_col: 검색할 질문이 들어있는 컬럼명 (예: "question")
    :param k_values: 검색할 k 값 리스트 (예: [1, 3, 5])
    """
    if isinstance(dataset, list):
        expected_contexts = [item.get(context_col, "") for item in dataset]
        queries = [item.get(query_col, "") for item in dataset]
    elif isinstance(dataset, dict):
        if context_col in dataset and query_col in dataset:
            expected_contexts = dataset[context_col]
            queries = dataset[query_col]
    else:
        raise ValueError("지원되지 않는 데이터셋 유형입니다.")

    hit_rates = []
    total_queries = len(queries)

    for k in k_values:
        retriever = chroma.as_retriever(search_kwargs={"k": k})
        hit_count = 0

        for query, expected_context in tqdm(zip(queries, expected_contexts)):
            retrieved_docs = retriever.invoke(query)
            retrieved_contexts = [doc.page_content for doc in retrieved_docs]

            if expected_context.lower() in [doc.lower() for doc in retrieved_contexts]:
                hit_count += 1

        hit_rate = hit_count / total_queries
        hit_rates.append(hit_rate)
    return hit_rates

In [ ]:
hitratio_calculator(
    chroma=chroma,
    dataset=benchmark_data,
    query_col="question",
    context_col="context",
    k_values=[1,3,5]
)

## bge-m3

In [14]:
import json
import chromadb
from langchain_openai.embeddings import OpenAIEmbeddings
from langchain_chroma import Chroma
from uuid import uuid4
import markdown2
from transformers import AutoTokenizer
import os
from concurrent.futures import ThreadPoolExecutor
from tqdm.autonotebook import tqdm

In [32]:
client = chromadb.HttpClient(
    host="3.35.104.197",
    port=10090
    )


In [ ]:
embedding_function = OpenAIEmbeddings(
    base_url = "https://16ttm8p7qmhnz9-10060.proxy.runpod.net/v1",
    model = "bge-m3",
    api_key="team3",
    tiktoken_enabled=False,
    embedding_ctx_length = 8100
    )

chroma = Chroma(collection_name="bge-m3", 
                embedding_function=embedding_function, 
                client=client)


In [ ]:
def insert_db(folder_path, chroma, tokenizer_model,  max_tokens=None, batch_size = 16):
    """
    ChromaDB 서버에 문서를 넣는 함수

    :param folderpath: 문서들이 있는 가장 상위 폴더 경로
    :param chroma: ChromaDB 클라이언트 객체
    :param tokenizer_model: 토크나이저 모델이름
    :param max_tokens:  None - 전체저장
                        max_tokens=512 - 512토큰단위 청킹 적용
    """
    texts=[]
    metadatas=[]
    tokenizer = AutoTokenizer.from_pretrained(tokenizer_model)
    for root, _, files in os.walk(folder_path):
        for filename in files:
            if filename.endswith((".txt", ".md", ".json")):
                file_path = os.path.join(root, filename)
    
                with open(file_path, "r", encoding="utf-8") as file:
                    content = file.read().strip()
    
                    if filename.endswith(".md"):
                        content = markdown2.markdown(content)
                    
                    elif filename.endswith(".json"):
                        try:
                            json_data = json.loads(content)
                            
                            if isinstance(json_data, dict):
                                filename = json_data.get("title", filename)
                            elif isinstance(json_data,list):
                                filename = " ".join(str(item.get("title", "")) for item in json_data if isinstance(item, dict))
            
                            if isinstance(json_data, list):
                                content = " ".join(str(item.get("content", "")) for item in json_data if isinstance(item, dict))
                            elif isinstance(json_data, dict):
                                content = str(json_data.get("content", ""))
                            else:
                                continue

                        except json.JSONDecodeError:
                            continue

                    texts.append(content)
                    metadatas.append({
                        "source": filename,
                        "doc_id": str(uuid4())
                    })

    num_batches = len(texts) // batch_size +1 if len(texts) % batch_size else len(texts) // batch_size
    for i in tqdm(range(5600, num_batches)):
        batch_text, batch_metadata = texts[i*batch_size:(i+1)*batch_size], metadatas[i*batch_size:(i+1)*batch_size]          
        chroma.add_texts(texts=batch_text, metadatas=batch_metadata)

In [ ]:
insert_db(
    folder_path = r"C:\Users\sang\Desktop\Final_Project\DB",
    chroma = chroma,
    tokenizer_model= "bge-m3",
    max_tokens = 8100,
    batch_size=4
)

In [ ]:
def hitratio_calculator(chroma, dataset, query_col, context_col, k_values):
    """
    ChromaDB 서버에서 검색 후 Hit Ratio를 계산하는 함수 (배치 처리 제거)

    :param chroma: ChromaDB 클라이언트 객체
    :param dataset: 데이터셋 (list of dicts)
    :param context_col: 정답이 들어있는 컬럼명 (예: "context")
    :param query_col: 검색할 질문이 들어있는 컬럼명 (예: "question")
    :param k_values: 검색할 k 값 리스트 (예: [1, 3, 5])
    """
    if isinstance(dataset, list):
        expected_contexts = [item.get(context_col, "") for item in dataset]
        queries = [item.get(query_col, "") for item in dataset]
    elif isinstance(dataset, dict):
        if context_col in dataset and query_col in dataset:
            expected_contexts = dataset[context_col]
            queries = dataset[query_col]
    else:
        raise ValueError("지원되지 않는 데이터셋 유형입니다.")
    
    def remove_all_special_chars(text):
        return re.sub(r"[^\w\s]", "", text, flags=re.UNICODE) 
    
    hit_rates = []
    total_queries = len(queries)

    for k in k_values:
        retriever = chroma.as_retriever(search_kwargs={"k": k})
        hit_count = 0

        for query, expected_context in tqdm(zip(queries, expected_contexts)):
            retrieved_docs = retriever.invoke(query)
            retrieved_contexts = [doc.page_content for doc in retrieved_docs]

            if remove_all_special_chars(expected_context) == remove_all_special_chars(retrieved_contexts[0]):
                hit_count += 1
            
        hit_rate = hit_count / total_queries
        hit_rates.append(hit_rate)
        print(f"k={k}, Hit Ratio = {hit_rate:.4f}")

    return hit_rates


In [43]:
with open(r"C:\Users\sang\Desktop\Final_Project\benchmark_data_1000_new.json", "r", encoding="utf-8" )as f:
    benchmark_data = json.load(f)

In [ ]:
hitratio_calculator(
    chroma=chroma,
    dataset=benchmark_data,
    query_col="question",
    context_col="context",
    k_values=[1,3,5]
)

900it [20:40,  1.38s/it]


k=1, Hit Ratio = 0.5911


900it [20:02,  1.34s/it]


k=3, Hit Ratio = 0.5911


900it [19:59,  1.33s/it]

k=5, Hit Ratio = 0.5911


[0.5911111111111111, 0.5911111111111111, 0.5911111111111111]

## HyQE

In [1]:
from tqdm import tqdm
import json
from langchain_openai.embeddings import OpenAIEmbeddings
import chromadb
from langchain_chroma import Chroma
from tqdm import tqdm
from transformers import AutoTokenizer
import os
import uuid

In [2]:
client = chromadb.HttpClient(
    host="3.35.104.197",
    port=10090
    )

In [3]:
embedding_function = OpenAIEmbeddings(
    base_url = "https://76mqtyy2wie64y-10050.proxy.runpod.net/v1",
    model = "multilingual-e5-large",
    api_key="team3",
    tiktoken_enabled=False,
    embedding_ctx_length = 502
)

chroma = Chroma(
                collection_name="hyqe", 
                embedding_function=embedding_function, 
                client=client
            )

In [ ]:
## insertt to chromaDB ##

with open("generated_questions.json", "r", encoding="utf-8") as f:
    dataset = json.load(f)

texts = []
metadatas = []

for data in dataset:
    texts.append(str(data["question"])) 
    metadatas.append({
        "original_content": data["context"],  
        "doc_id": str(uuid.uuid4())  
    })

batch_size = 16
num_batches = (len(texts) + batch_size - 1) // batch_size 

for i in tqdm(range(1249, num_batches), desc="Processing batches"):
    batch_text = texts[i * batch_size : (i + 1) * batch_size]  
    batch_metadata = metadatas[i * batch_size : (i + 1) * batch_size]

    chroma.add_texts(texts=batch_text, metadatas=batch_metadata)

print(f"{len(texts)}개의 문서")

In [6]:
chroma.get()

{'ids': ['d067cb0f-0da4-490c-91fe-d5cda1468455',
  'b84ef854-4f7b-4f0c-9a60-d1a74256c05d',
  '59a0a9ba-d2f3-4805-9096-09ab369f0057',
  '8594c083-ce5d-4d99-80d3-42bf138966c6',
  '71b2a04c-a69d-44e7-91b6-4767f49bfd81',
  '28fd84ec-6d1c-4b6d-b115-86150d6c6c54',
  '2c25fc98-27be-4a80-be76-2358a022b1e9',
  '10419c25-636d-482a-ae71-2b3bf7f4c02a',
  '6a376846-bbb1-4da7-a78e-6569f71d976f',
  '67c18bd5-dbac-4d0e-9a06-b01b0fef28db',
  '28745199-d4ed-4fc1-977d-c5636660619f',
  '1bf60bbc-6bd3-4f45-904d-42231d13dd86',
  '59a28655-4b64-4a58-887b-c97150f1c856',
  'c3c0c990-4aff-45f9-a3dd-1a0279d7b3a0',
  'c5e50f7e-fc33-4e50-ba86-479a65e19c41',
  '6b0b5d44-c39d-4565-b3e7-09c16daf8ac7',
  'a33f76e7-ab58-41c9-b7b8-158e5bb87b2b',
  '11f22c4e-996b-445c-8b22-3d2575546709',
  '4eaa9eee-6b46-4bc9-a781-c266f62f50a2',
  '5eb51edb-7d9b-4ef5-9671-0d67640ed743',
  '97056704-aea0-4a92-a074-af6981711a1e',
  'abf325f3-e16a-4e96-9292-2ddade6e411a',
  'b537f423-eed1-4aeb-a68e-71c017bd939f',
  'aa337681-fdea-4a2b-b2f2-

In [ ]:
## k=1 일때

import pandas as pd
expected_contexts = [item.get("context") for item in benchmark_data]
queries = [item.get("question") for item in benchmark_data]

f=[]
hit_count=0
k = 3
retriever = chroma.as_retriever(search_kwargs={"k": k})

for query, expected_context in tqdm(zip(queries, expected_contexts)):
    retrieved_docs = retriever.invoke(query)
    retrieved_contexts = [doc.metadata["original_content"] for doc in retrieved_docs if "original_content" in doc.metadata]
    generated_quesitons = [doc.page_content for doc in retrieved_docs]
    
    if expected_context.lower() in [doc.lower() for doc in retrieved_contexts]:
            hit_count += 1
    else:
        f.append([query, expected_context, retrieved_contexts, generated_quesitons])

    hit_rate = hit_count / len(queries)

    df= pd.DataFrame(f, columns=["질문", "정답", "retrieve_context", "생성된질문들"])
    df.to_csv(f"false_list_k_{k}.csv",encoding="utf-8", index=False)

hit_rate

In [ ]:
## k=3 일 때 

import pandas as pd
expected_contexts = [item.get("context") for item in benchmark_data]
queries = [item.get("question") for item in benchmark_data]

f=[]
hit_count=0
k = 3
retriever = chroma.as_retriever(search_kwargs={"k": k})

for query, expected_context in tqdm(zip(queries, expected_contexts)):
    retrieved_docs = retriever.invoke(query)
    retrieved_contexts = [doc.metadata["original_content"] for doc in retrieved_docs if "original_content" in doc.metadata]
    generated_quesitons = [doc.page_content for doc in retrieved_docs]
    
    if expected_context.lower() in [doc.lower() for doc in retrieved_contexts]:
            hit_count += 1
    else:
        f.append([query, expected_context, retrieved_contexts, generated_quesitons])

    hit_rate = hit_count / len(queries)


    df= pd.DataFrame(f, columns=["질문", "정답", "retrieve_context", "생성된질문들"])
    df.to_csv(f"false_list_k_{k}.csv",encoding="utf-8", index=False)

hit_rate

In [ ]:
## k=5 일 때 

import pandas as pd
expected_contexts = [item.get("context") for item in benchmark_data]
queries = [item.get("question") for item in benchmark_data]

f=[]
hit_count=0
k = 3
retriever = chroma.as_retriever(search_kwargs={"k": k})

for query, expected_context in tqdm(zip(queries, expected_contexts)):
    retrieved_docs = retriever.invoke(query)
    retrieved_contexts = [doc.metadata["original_content"] for doc in retrieved_docs if "original_content" in doc.metadata]
    generated_quesitons = [doc.page_content for doc in retrieved_docs]
    
    if expected_context.lower() in [doc.lower() for doc in retrieved_contexts]:
            hit_count += 1
    else:
        f.append([query, expected_context, retrieved_contexts, generated_quesitons])

    hit_rate = hit_count / len(queries)

    df= pd.DataFrame(f, columns=["질문", "정답", "retrieve_context", "생성된질문들"])
    df.to_csv(f"false_list_k_{k}.csv",encoding="utf-8", index=False)

hit_rate

In [ ]:
embedding_function = OpenAIEmbeddings(
    base_url = "https://76mqtyy2wie64y-10050.proxy.runpod.net/v1",
    model = "multilingual-e5-large",
    api_key="team3",
    tiktoken_enabled=False,
    embedding_ctx_length = 502
)

chroma = Chroma(
                collection_name="hyqe", 
                embedding_function=embedding_function, 
                client=client
            )

In [ ]:
with open(r"C:\Users\sang\Desktop\Final_Project\benchmark_data_1000_new.json", "r", encoding="utf-8" )as f:
    benchmark_data = json.load(f)

# Collection 생성

In [2]:
import json
import chromadb
from langchain_openai.embeddings import OpenAIEmbeddings
from langchain_chroma import Chroma
from uuid import uuid4
import markdown2
from transformers import AutoTokenizer
import os
from concurrent.futures import ThreadPoolExecutor
from tqdm.autonotebook import tqdm

client = chromadb.HttpClient(
    host="3.35.104.197",
    port=10090
)

embedding_function = OpenAIEmbeddings(
    base_url = "https://76mqtyy2wie64y-10050.proxy.runpod.net/v1",
    model = "multilingual-e5-large",
    api_key="team3",
    tiktoken_enabled=False,
    embedding_ctx_length = 502
)

## 1. Brand collection

In [ ]:
## brand_collection

client = chromadb.HttpClient(
    host="3.35.104.197",
    port=10090
)

embedding_function = OpenAIEmbeddings(
    base_url = "https://76mqtyy2wie64y-10050.proxy.runpod.net/v1",
    model = "multilingual-e5-large",
    api_key="team3",
    tiktoken_enabled=False,
    embedding_ctx_length = 502
)
chroma = Chroma(
    collection_name = "brand",
    embedding_function=embedding_function,
    client=client
)

texts = []
metadatas = []

with open("brand.json", "r", encoding="utf-8")as f:
    brand = json.load(f)

for item in tqdm(brand):
    info =item.get("brand_info", "")
    eng = item.get("brand_eng")
    kor = item.get("brand_kor")
    subtitle = item.get("brand_subtitle")
    ceo = item.get("ceo")
    founded_year = item.get("founded_year")
    
    context = (
        f"브랜드 설명: {info},\n"
        f"영어 이름: {eng},\n"
        f"한글 이름: {kor},\n"
        f"브랜드 subtitle: {subtitle},\n"
        f"영어 이름: {eng},\n"
        f"CEO: {ceo},\n"
        f"설립연도: {founded_year}"
        )
    
    texts.append(context)
    metadatas.append({
            "company_id" : item.get("company_id"),
            "brand_id": item.get("brand_id"),
            "doc_id" : str(uuid4()),
            "scope" : str("INNER"),
            "questions": " "
    })
chroma.add_texts(texts=texts, metadatas=metadatas)

In [ ]:
embedding_function = OpenAIEmbeddings(
    base_url = "https://76mqtyy2wie64y-10050.proxy.runpod.net/v1",
    model = "multilingual-e5-large",
    api_key="team3",
    tiktoken_enabled=False,
    embedding_ctx_length = 502
)

## 2. Posts collection

In [ ]:
## post collection

chroma = Chroma(
                collection_name="posts", 
                embedding_function=embedding_function, 
                client=client
            )

with open("posts.json", "r", encoding="utf-8")as f:
    posts = json.load(f)

with open("posts.json", "r", encoding="utf-8")as f:
    posts = json.load(f)
batch_size = 160
texts = []
metadatas = []

for post in tqdm(posts):
    page_content = post.get("content", None)
    
    texts.append(page_content)
    metadatas.append({
        "post_id": post.get("post_id"),
        "post_type":post.get("post_type"),
        "post_ctgry": post.get("post_ctgry"),
        "title": post.get("title"),
        "company_id": post.get("company_id"),
        "document_id": post.get("document_id"),
        "authority_level": post.get("authority_level"),
        "scope" : post.get("scope"),
        "doc_id":str(uuid4()),
        "questions": ""
    })
    
num_batches = len(texts) // batch_size +1 if len(texts) % batch_size else len(texts) // batch_size
for i in tqdm(range(num_batches)):
    batch_text, batch_metadata = texts[i*batch_size:(i+1)*batch_size], metadatas[i*batch_size:(i+1)*batch_size]           
    chroma.add_texts(texts=batch_text, metadatas=batch_metadata)

## 3. Ingredient collection

In [ ]:
## ingredient_collection

import json

chroma = Chroma(
    collection_name = "ingredient",
    embedding_function=embedding_function,
    client=client
)

texts=[]
metadatas=[]

with open("ingredient.json", "r", encoding="utf-8")as f:
    ingredient = json.load(f)

for item in tqdm(ingredient[5088:]):
    cas_no =item.get("cas_no")
    ingred_eng = item.get("ingred_eng")
    ingred_kor = item.get("ingred_kor")
    definition = item.get("definition")
    synonym = item.get("synonym")
    uses = item.get("uses")
    caution = item.get("caution")
    effect = item.get("effect")
    
    context = (
        f"CAS No: {cas_no},\n"
        f"영어 이름: {ingred_eng},\n"
        f"한글 이름: {ingred_kor},\n"
        f"정의: {definition},\n"
        f"유의어: {synonym},\n"
        f"용도: {uses},\n"
        f"주의사항: {caution}\n"
        f"효과: {effect}"
        )
    
    texts.append(context)
    metadatas.append({
            "ingred_id" : item.get("ingred_id"),
            "doc_id" : str(uuid4())
    })

batch_size = 16
num_batches = len(texts) // batch_size +1 if len(texts) % batch_size else len(texts) // batch_size
for i in tqdm(range(num_batches)):
    batch_text, batch_metadata = texts[i*batch_size:(i+1)*batch_size], metadatas[i*batch_size:(i+1)*batch_size]             
    chroma.add_texts(texts=batch_text, metadatas=batch_metadata)

## 4. Cosmetic collection

In [ ]:
## cosmetic collection에 match
import json

chroma = Chroma(
    collection_name = "cosmetic",
    embedding_function=embedding_function,
    client=client
)

texts=[]
metadatas=[]

with open("cosmetic_all.json", "r", encoding="utf-8")as f:
    cosmetic = json.load(f)

for item in cosmetic:
    if item.get("brand_id") is None:
        item["brand_id"] = "None"
    cosmetic_id =item.get("cosmetic_id", "None")
    category_1 = item.get("category_1", "None")
    category_2 = item.get("category_2", "None")
    product_name = item.get("product_name", "None")
    product_info = item.get("product_info", "None")
    capacity = item.get("capacity", "None")
    specification = item.get("specification", "None")
    expiration_date = item.get("expiration_date", "None")
    use_period = item.get("use_period", "None")
    ingredients = item.get("ingredients", "None")
    precaution = item.get("precaution", "None")
    price = item.get("price", "None")
    manufacture = item.get("manufacture", "None")
    quality_standards = item.get("quality_standards", "None")
    mfds = item.get("mfds", "None")
    brand_kor = item.get("brand_kor", "None")
    brand_id = item.get("brand_id", "None")
    image_url = item.get("image_url", "None")

    content = (
        f"cosmetic_id: {cosmetic_id},\n"
        f"category_1: {category_1},\n"
        f"category_2: {category_2},\n"
        f"제품이름: {product_name},\n"
        f"브랜드: {brand_kor},\n"
        f"제품정보: {product_info},\n"
        f"가격: {price}\n"
        f"용도: {capacity},\n"
        f"사양: {specification}\n"
        f"기한: {expiration_date}\n"
        f"사용법: {use_period}\n"
        f"성분: {ingredients}\n"
        f"주의사항: {precaution}\n"
        f"제조사: {manufacture}\n"
        f"주의사항: {quality_standards}\n"
        )
    
    texts.append(content)
    metadatas.append({
            "cosmetic_id" : cosmetic_id,
            "brand_id" : brand_id,
            "category_1": category_1,
            "category_2": category_2,
            "image_url": image_url,
            "scope" : item.get("scope", "None"),        
            "doc_id" : str(uuid4())
            })
    
batch_size = 16
num_batches = len(texts) // batch_size +1 if len(texts) % batch_size else len(texts) // batch_size
for i in tqdm(range(49, num_batches)):
    batch_text, batch_metadata = texts[i*batch_size:(i+1)*batch_size], metadatas[i*batch_size:(i+1)*batch_size]              
    chroma.add_texts(texts=batch_text, metadatas=batch_metadata)

## HyQE로 생성한 질문들 metadata에 매칭(jaccard 유사도)

In [ ]:
## posts 콜렉션 match

import json, re
import pandas as pd
from pprint import pprint
with open("generated_questions.json", "r", encoding="utf-8")as f:
    a = json.load(f)
generated_questions = a[22286:]

hit_count = 0
f=[]
a=[]
contexts = [item.get("context") for item in generated_questions]
questions = [item.get("question") for item in generated_questions]
retriever = chroma.as_retriever(search_kwargs={"k": 1})
collection = client.get_collection("posts")

def jaccard_similarity(list1, list2):
    """자카드 유사도 함수"""
    s1 = set(list1)
    s2 = set(list2)
    return float(len(s1.intersection(s2)) / len(s1.union(s2)))

def clean_text(text):
    """텍스트에서 특수문자, 줄바꿈, 특정 기호 제거"""
    text = re.sub(r"[\n=#]", "", text)  
    text = re.sub(r"[^\w\s]", "", text, flags=re.UNICODE) 
    return text.lower().strip() 

for context, question in tqdm(zip(contexts, questions), total=len(contexts)):
    retrieved_docs = retriever.invoke(context)
    retrieved_contents = [doc.page_content for doc in retrieved_docs]

    for retrieved_content in retrieved_contents:
        jaccard_score = jaccard_similarity(context, retrieved_content)

## 자카드 유사도 0.9 이상의 질문만 매칭
        for doc in retrieved_docs:
            if jaccard_score >= 0.9: 
                hit_count += 1
                a.append([context, retrieved_content, question])         
                doc.metadata["questions"] = question
                collection.update(
                    ids=[doc.id],
                    metadatas=[doc.metadata]
                )
            else:
                f.append([context, retrieved_content, question])
            df_a = pd.DataFrame(a, columns=["context", "retrieve_context", "생성된질문들"])
            df_a.to_csv(f"true_list.csv",encoding="utf-8", index=False)
            df_f= pd.DataFrame(f, columns=["context", "retrieve_context", "생성된질문들"])
            df_f.to_csv(f"false_list.csv",encoding="utf-8", index=False)

hit_rate = hit_count / len(contexts)
print(f"Hit Ratio = {hit_rate:.4f}")

In [ ]:
## cosmeitc 콜렉션 match

chroma = Chroma(
    collection_name = "cosmetic",
    embedding_function=embedding_function,
    client=client
)
import json, re
import pandas as pd
from pprint import pprint

with open("generated_questions.json", "r", encoding="utf-8")as f:
    a = json.load(f)
generated_questions = a[30:869]
# generated_questions = a[30:45]
# generated_questions = a[100:105]

hit_count = 0
f=[]
a=[]
contexts = [item.get("context") for item in generated_questions]
questions = [item.get("question") for item in generated_questions]
retriever = chroma.as_retriever(search_kwargs={"k": 1})
collection = client.get_collection("cosmetic")

def jaccard_similarity(list1, list2):
    """자카드 유사도 함수"""
    s1 = set(list1)
    s2 = set(list2)
    return float(len(s1.intersection(s2)) / len(s1.union(s2)))

def clean_text(text):
    """텍스트에서 특수문자, 줄바꿈, 특정 기호 제거"""
    text = re.sub(r"[\n=#]", "", text)  # \n, =, # → 공백으로 변환
    text = re.sub(r"[^\w\s]", "", text, flags=re.UNICODE)  # 나머지 특수문자 제거
    return text.lower().strip()  # 소문자로 변환 및 공백 제거

for context, question in tqdm(zip(contexts, questions), total=len(contexts)):
    retrieved_docs = retriever.invoke(context)
    retrieved_contents = [doc.page_content for doc in retrieved_docs]

    for retrieved_content in retrieved_contents:
        jaccard_score = jaccard_similarity(context, retrieved_content)
        
        for doc in retrieved_docs:
            if jaccard_score >= 0.5:
                hit_count += 1
                a.append([jaccard_score, context, retrieved_content, question])         
                doc.metadata["questions"] = question # if question not in doc.metadata["questions"])
                collection.update(
                    ids=[doc.id],
                    metadatas=[doc.metadata]
                )
                
            else:
                f.append([jaccard_score, context, retrieved_content, question])
                print(f"{jaccard_score}")
            
            df_t = pd.DataFrame(a, columns=["jaccard_score", "context", "retrieve_context", "생성된질문들"])
            df_t.to_csv(f"true_list_cosmetic.csv",encoding="utf-8", index=False)
            df_f= pd.DataFrame(f, columns=["jaccard_score", "context", "retrieve_context", "생성된질문들"])
            df_f.to_csv(f"false_list_cosmetic.csv",encoding="utf-8", index=False)

hit_rate = hit_count / len(contexts)
print(f"Hit Ratio = {hit_rate:.4f}")

In [ ]:
## brand 콜렉션 match

chroma = Chroma(
    collection_name = "brand",
    embedding_function=embedding_function,
    client=client
)
import json, re
import pandas as pd
from pprint import pprint

with open("generated_questions.json", "r", encoding="utf-8")as f:
    a = json.load(f)
generated_questions = a[0:30]

hit_count = 0
f=[]
a=[]
contexts = [item.get("context") for item in generated_questions]
questions = [item.get("question") for item in generated_questions]
retriever = chroma.as_retriever(search_kwargs={"k": 1})
collection = client.get_collection("cosmetic")

def jaccard_similarity(list1, list2):
    """자카드 유사도 함수"""
    s1 = set(list1)
    s2 = set(list2)
    return float(len(s1.intersection(s2)) / len(s1.union(s2)))

def clean_text(text):
    """텍스트에서 특수문자, 줄바꿈, 특정 기호 제거"""
    text = re.sub(r"[\n=#]", "", text)  # \n, =, # → 공백으로 변환
    text = re.sub(r"[^\w\s]", "", text, flags=re.UNICODE)  # 나머지 특수문자 제거
    return text.lower().strip()  # 소문자로 변환 및 공백 제거

for context, question in tqdm(zip(contexts, questions), total=len(contexts)):
    retrieved_docs = retriever.invoke(context)
    retrieved_contents = [doc.page_content for doc in retrieved_docs]

    for retrieved_content in retrieved_contents:
        jaccard_score = jaccard_similarity(context, retrieved_content)
        
        for doc in retrieved_docs:
            if jaccard_score >= 0.5:
                hit_count += 1
                a.append([jaccard_score, context, retrieved_content, question])         
                doc.metadata["questions"] = question # if question not in doc.metadata["questions"])
                collection.update(
                    ids=[doc.id],
                    metadatas=[doc.metadata]
                )
                
            else:
                f.append([jaccard_score, context, retrieved_content, question])
                print(f"{jaccard_score}")
            
            df_t = pd.DataFrame(a, columns=["jaccard_score", "context", "retrieve_context", "생성된질문들"])
            df_t.to_csv(f"true_list_cosmetic.csv",encoding="utf-8", index=False)
            df_f= pd.DataFrame(f, columns=["jaccard_score", "context", "retrieve_context", "생성된질문들"])
            df_f.to_csv(f"false_list_cosmetic.csv",encoding="utf-8", index=False)

hit_rate = hit_count / len(contexts)
print(f"Hit Ratio = {hit_rate:.4f}")

In [ ]:
## ingredient collection match

chroma = Chroma(
    collection_name = "ingredient",
    embedding_function=embedding_function,
    client=client
)
import json, re
import pandas as pd
from pprint import pprint

with open("generated_questions.json", "r", encoding="utf-8")as f:
    a = json.load(f)
# generated_questions = a[870:22185]
# generated_questions = a[2163:22185]
# generated_questions = a[3253:22185] # not match : 0개
# generated_questions = a[6017:22185]  # not match : 12개
# generated_questions = a[13673:22185] # not match : 8개
# generated_questions = a[16669:22185] # not match : 0개
generated_questions = a[17434:22185]
# generated_questions = a[870:900]
hit_count = 0
f=[]
a=[]
contexts = [item.get("context") for item in generated_questions]
questions = [item.get("question") for item in generated_questions]
retriever = chroma.as_retriever(search_kwargs={"k": 1})
collection = client.get_collection("ingredient")

def jaccard_similarity(list1, list2):
    """자카드 유사도 함수"""
    s1 = set(list1)
    s2 = set(list2)
    return float(len(s1.intersection(s2)) / len(s1.union(s2)))

def clean_text(text):
    """텍스트에서 특수문자, 줄바꿈, 특정 기호 제거"""
    text = re.sub(r"[\n=#]", "", text)  # \n, =, # → 공백으로 변환
    text = re.sub(r"[^\w\s]", "", text, flags=re.UNICODE)  # 나머지 특수문자 제거
    return text.lower().strip()  # 소문자로 변환 및 공백 제거

for context, question in tqdm(zip(contexts, questions), total=len(contexts)):
    retrieved_docs = retriever.invoke(context)
    retrieved_contents = [doc.page_content for doc in retrieved_docs]

    for retrieved_content in retrieved_contents:
        jaccard_score = jaccard_similarity(context, retrieved_content)
        
        for doc in retrieved_docs:
            if jaccard_score >= 0.5:
                hit_count += 1
                a.append([jaccard_score, context, retrieved_content, question])         
                doc.metadata["questions"] = question # if question not in doc.metadata["questions"])
                collection.update(
                    ids=[doc.id],
                    metadatas=[doc.metadata]
                )
                
            else:
                f.append([jaccard_score, context, retrieved_content, question])
                print(f"{jaccard_score}")

            df_t = pd.DataFrame(a, columns=["jaccard_score", "context", "retrieve_context", "생성된질문들"])
            df_t.to_csv(f"true_list_ingredient.csv",encoding="utf-8", index=False)
            df_f= pd.DataFrame(f, columns=["jaccard_score", "context", "retrieve_context", "생성된질문들"])
            df_f.to_csv(f"false_list_ingredient.csv",encoding="utf-8", index=False)

hit_rate = hit_count / len(contexts)
print(f"Hit Ratio = {hit_rate:.4f}")